# KOHLER AI Bathroom Designer : Data & Recommendation Pipeline

This notebook documents the data processing and recommendation pipeline used by the KOHLER AI Bathroom Designer.

The pipeline uses the curated KOHLER product catalog and applies budget, bathroom-size, style, water-efficiency, luxury, and compactness considerations to generate product recommendations and complete bathroom combinations.

## 1. Objective

The recommendation pipeline supports two request types:

- **Complete Bathroom:** select one Toilet, Faucet, Shower, and Vanity while satisfying the supplied budget and bathroom dimensions.
- **Specific Products:** search for products from the categories requested by the user.

The pipeline is deterministic after the requirements have been structured. The LLM requirement-extraction component is separate from this notebook.

## 2. Imports and Dataset Path

In [ ]:
import pandas as pd
from pathlib import Path

DATA_PATH = Path("../data/KOHLER_AI_Bathroom_Designer_FINAL_DATASET_v3.csv")

if not DATA_PATH.exists():
    DATA_PATH = Path("data/KOHLER_AI_Bathroom_Designer_FINAL_DATASET_v3.csv")

products = pd.read_csv(DATA_PATH)
print(f"Dataset path: {DATA_PATH.resolve()}")
print(f"Rows: {len(products)}")
print(f"Columns: {len(products.columns)}")

## 3. Dataset Inspection

The recommendation engine uses product information such as category, price, dimensions, style, water usage, and luxury-related attributes.

In [ ]:
display(products.head())
print("\nColumns:")
print(products.columns.tolist())

In [ ]:
print("Missing values:")
display(products.isna().sum().sort_values(ascending=False).to_frame("missing"))

In [ ]:
def filter_by_size(products, max_width_cm, max_depth_cm):
    return products[
        products["width_cm"].notna()
        & products["depth_cm"].notna()
        & (products["width_cm"] <= max_width_cm)
        & (products["depth_cm"] <= max_depth_cm)
    ].copy()

bathroom_width_ft = 8
bathroom_depth_ft = 10
max_width_cm = bathroom_width_ft * 30.48
max_depth_cm = bathroom_depth_ft * 30.48

size_filtered = filter_by_size(products, max_width_cm, max_depth_cm)

print(f"Bathroom: {bathroom_width_ft} ft × {bathroom_depth_ft} ft")
print(f"Dimension-compatible products: {len(size_filtered)}")

## 4. Basic Data Preparation

The recommendation engine keeps products with usable price and dimension information when applying the corresponding hard constraints.

In [ ]:
print("Price summary:")
display(products["price"].describe())

print("\nDimension summary:")
display(products[["width_cm", "depth_cm"]].describe())

## 5. Hard Constraint — Budget

Products whose price exceeds the user's maximum budget are removed.

In [ ]:
def filter_products(products, budget):
    return products[
        products["price"].notna()
        & (products["price"] <= budget)
    ].copy()

example_budget = 150000
budget_filtered = filter_products(products, example_budget)
print(f"Products within ₹{example_budget:,.0f}: {len(budget_filtered)}")

## 6. Hard Constraint : Bathroom Dimensions

Bathroom dimensions are supplied in feet by the application and converted to centimeters using:

`1 ft = 30.48 cm`

The current recommendation filter requires each product's catalog width and depth to fit within the supplied maximum dimensions.

In [ ]:
def filter_by_size(products, max_width_cm, max_depth_cm):
    return products[
        products["width_cm"].notna()
        & products["depth_cm"].notna()
        & (products["width_cm"] <= max_width_cm)
        & (products["depth_cm"] <= max_depth_cm)
    ].copy()

bathroom_width_ft = 8
bathroom_depth_ft = 10
max_width_cm = bathroom_width_ft * 30.48
max_depth_cm = bathroom_depth_ft * 30.48

size_filtered = filter_by_size(products, max_width_cm, max_depth_cm)
            
print(f"Bathroom: {bathroom_width_ft} ft × {bathroom_depth_ft} ft")
print(f"Dimension-compatible products: {len(size_filtered)}")

## 7. Style Scoring

Supported styles are **Modern, Minimalist, Luxury, Traditional, and Japanese Zen**. Exact matches receive the highest style score, while related styles receive intermediate scores.

In [ ]:
def calculate_style_score(product_style, desired_style):
    if desired_style is None:
        return 5

    if product_style == desired_style:
        return 10

    if desired_style == "Japanese Zen":
        if product_style == "Organic": return 9
        if product_style == "Minimalist": return 8
        if product_style == "Modern": return 6
        return 5

    if desired_style == "Modern":
        if product_style == "Minimalist": return 8
        if product_style == "Luxury": return 7
        if product_style == "Organic": return 6
        if product_style == "Traditional": return 5
        return 5

    if desired_style == "Minimalist":
        if product_style == "Modern": return 8
        if product_style == "Organic": return 7
        if product_style == "Luxury": return 5
        if product_style == "Traditional": return 5
        return 5

    if desired_style == "Luxury":
        if product_style == "Modern": return 7
        if product_style == "Traditional": return 6
        if product_style == "Organic": return 5
        if product_style == "Minimalist": return 5
        return 5

    return 5

## 8. Water Efficiency Score

For Toilet, Faucet, Shower, and Bathtub products with available water-usage data, lower water usage receives a higher score on a 1–10 scale. Products with missing water usage receive the default internal score of 5 unless they are excluded for a Water Saving request.

In [ ]:
def calculate_water_score(products):
    products = products.copy()
    products["water_score"] = 5.0

    water_categories = ["Toilet", "Faucet", "Shower", "Bathtub"]

    for category in water_categories:
        category_mask = (
            (products["category"] == category)
            & products["water_usage"].notna()
        )
        valid_usage = products.loc[category_mask, "water_usage"]

        if valid_usage.empty:
            continue

        min_usage = valid_usage.min()
        max_usage = valid_usage.max()

        if max_usage == min_usage:
            products.loc[category_mask, "water_score"] = 10.0
        else:
            products.loc[category_mask, "water_score"] = (
                10 - ((products.loc[category_mask, "water_usage"] - min_usage)
                      / (max_usage - min_usage)) * 9
            )

    return products

products_scored = calculate_water_score(products)
display(products_scored[["category", "water_usage", "water_score"]].head(10))

## 9. Compactness Score

Product footprint is calculated as:

`footprint = width_cm × depth_cm`

Within each relevant category, smaller footprints receive higher compactness scores.

In [ ]:
def calculate_compact_score(products):
    products = products.copy()
    products["footprint"] = products["width_cm"] * products["depth_cm"]
    products["compact_score"] = 5.0

    categories = ["Toilet", "Faucet", "Shower", "Bathtub", "Vanity"]

    for category in categories:
        category_mask = (
            (products["category"] == category)
            & products["footprint"].notna()
        )
        valid_footprints = products.loc[category_mask, "footprint"]

        if valid_footprints.empty:
            continue

        min_footprint = valid_footprints.min()
        max_footprint = valid_footprints.max()

        if max_footprint == min_footprint:
            products.loc[category_mask, "compact_score"] = 10.0
        else:
            products.loc[category_mask, "compact_score"] = (
                10 - ((products.loc[category_mask, "footprint"] - min_footprint)
                      / (max_footprint - min_footprint)) * 9
            )

    return products

products_scored = calculate_compact_score(products_scored)
display(products_scored[["category", "width_cm", "depth_cm", "footprint", "compact_score"]].head(10))

## 10. Priority-Based Product Score

The final internal product score combines style, water efficiency, luxury, and compactness.

| Priority | Style | Water | Luxury | Compactness |
|---|---:|---:|---:|---:|
| Water Saving | 40% | 40% | — | 20% |
| Luxury | 40% | 20% | 40% | — |
| Best Overall | 50% | 25% | 15% | 10% |

In [ ]:
def calculate_score(product, desired_style, priority):
    style_score = calculate_style_score(product["style"], desired_style)
    water_score = product["water_score"]
    luxury_score = product["luxury_score"]
    compact_score = product["compact_score"]

    if priority == "Water Saving":
        return style_score * 0.4 + water_score * 0.4 + compact_score * 0.2
    elif priority == "Luxury":
        return style_score * 0.4 + luxury_score * 0.4 + water_score * 0.2
    else:
        return style_score * 0.5 + water_score * 0.25 + luxury_score * 0.15 + compact_score * 0.10

## 11. Example Product Ranking

This example applies the same scoring logic used by the recommendation engine.

In [ ]:
desired_style = "Modern"
priority = "Water Saving"

ranked_demo = calculate_compact_score(calculate_water_score(products)).copy()
ranked_demo["score"] = ranked_demo.apply(
    lambda p: calculate_score(p, desired_style, priority), axis=1
)

display(
    ranked_demo.sort_values("score", ascending=False)[
        ["category", "name", "price", "style", "water_score", "compact_score", "score"]
    ].head(10)
)

## 12. Pareto Pruning

For complete bathroom optimization, each state contains total cost, total score, and selected products. Pareto pruning removes states that do not improve the score frontier as cost increases.

In [ ]:
def prune_dominated_states(states):
    states = sorted(states, key=lambda state: (state[0], -state[1]))
    pruned_states = []
    best_score_seen = -1

    for cost, score, selected_products in states:
        if score > best_score_seen:
            pruned_states.append((cost, score, selected_products))
            best_score_seen = score

    return pruned_states

## 13. Complete Bathroom Optimization

A complete bathroom recommendation contains exactly one product from each required category:

- Toilet
- Faucet
- Shower
- Vanity

The optimizer keeps combinations whose total cost does not exceed the budget and ranks valid states by total score, using lower cost as the tie-breaker.

In [ ]:
def optimize_bathroom_top_k(products, desired_style, priority, budget, top_k=3):
    categories = ["Toilet", "Faucet", "Shower", "Vanity"]
    top_k = max(1, int(top_k))
    products = products.copy()

    if priority == "Water Saving":
        water_categories = ["Toilet", "Faucet", "Shower", "Bathtub"]
        unknown_water = (
            products["category"].isin(water_categories)
            & products["water_usage"].isna()
        )
        products = products[~unknown_water].copy()

    products = calculate_water_score(products)
    products = calculate_compact_score(products)
    products["score"] = products.apply(
        lambda product: calculate_score(product, desired_style, priority), axis=1
    )

    states = [(0, 0.0, [])]

    for category in categories:
        category_products = products[products["category"] == category]
        if category_products.empty:
            return []

        candidates = []
        for current_cost, current_score, selected in states:
            for _, product in category_products.iterrows():
                if pd.isna(product["price"]):
                    continue

                product_price = int(product["price"])
                new_cost = current_cost + product_price
                if new_cost > budget:
                    continue

                new_score = current_score + float(product["score"])
                candidates.append((new_cost, new_score, selected + [product]))

        if not candidates:
            return []

        states = prune_dominated_states(candidates)

    ranked_states = sorted(states, key=lambda state: (-state[1], state[0]))
    return [pd.DataFrame(selected_products) for _, _, selected_products in ranked_states[:top_k]]

## 14. Product Specific Search

When the user requests specific product categories rather than a complete bathroom, the engine filters by category and budget, calculates the same internal scores, and returns the top products for each requested category.

In [ ]:
def search_products(products, requested_categories, budget, desired_style, priority, top_n=5):
    products = products.copy()
    products = products[products["category"].isin(requested_categories)].copy()
    products = products[products["price"].notna() & (products["price"] <= budget)].copy()

    if priority == "Water Saving":
        water_categories = ["Toilet", "Faucet", "Shower", "Bathtub"]
        unknown_water = (
            products["category"].isin(water_categories)
            & products["water_usage"].isna()
        )
        products = products[~unknown_water].copy()

    if products.empty:
        return pd.DataFrame()

    products = calculate_water_score(products)
    products = calculate_compact_score(products)
    products["score"] = products.apply(
        lambda product: calculate_score(product, desired_style, priority), axis=1
    )
    products = products.sort_values("score", ascending=False)

    results = []
    for category in requested_categories:
        category_products = products[products["category"] == category].head(top_n)
        if not category_products.empty:
            results.append(category_products)

    return pd.concat(results, ignore_index=True) if results else pd.DataFrame()

## 15. Example : Complete Bathroom Recommendation

The following example mirrors one of the validation scenarios used by the project.

In [ ]:
example_requirements = {
    "budget": 200000,
    "desired_style": "Modern",
    "priority": "Water Saving",
    "bathroom_width_ft": 8,
    "bathroom_depth_ft": 9,
}

filtered = filter_products(products, example_requirements["budget"])
filtered = filter_by_size(
    filtered,
    example_requirements["bathroom_width_ft"] * 30.48,
    example_requirements["bathroom_depth_ft"] * 30.48,
)

bundles = optimize_bathroom_top_k(
    filtered,
    example_requirements["desired_style"],
    example_requirements["priority"],
    example_requirements["budget"],
    top_k=3,
)

print(f"Recommendation sets generated: {len(bundles)}")

for i, bundle in enumerate(bundles, start=1):
    total = bundle["price"].sum()
    print(f"\nOption {i}: ₹{total:,.0f}")
    display(bundle[["category", "name", "price", "width_cm", "depth_cm"]])

## 16. Example : Specific Product Search

In [ ]:
search_result = search_products(
    products=products,
    requested_categories=["Toilet"],
    budget=50000,
    desired_style="Modern",
    priority="Water Saving",
    top_n=5,
)

display(search_result[["category", "name", "price", "style", "water_score", "score"]])

## 17. Recommendation Summary

The pipeline follows this sequence:

```text
1. Product Catalog
2. Budget Filtering
3. Bathroom Size Filtering (complete bathroom)
4. Water Efficiency + Compactness Scores
5. Style + Priority-Based Scoring
6. Combination Optimization / Product Search
7. Pareto Pruning
8. Top-K Recommendations
```

The production application imports these functions from `src/recommender.py`; this notebook provides an inspectable demonstration of the core logic.

## 18. Relationship to the Application

The Gradio application collects requirements and presents the recommendations to the user. For natural-language input, Gemini converts the user's request into validated structured requirements. The recommendation engine then performs the deterministic filtering, scoring, and optimization described in this notebook.

The resulting complete bathroom bundle can also be passed to the separate layout module for 2D visualization.

## Conclusion

The notebook demonstrates the core recommendation pipeline behind the KOHLER AI Bathroom Designer: catalog inspection, hard-constraint filtering, feature scoring, priority-based ranking, Pareto pruning, and Top-K bathroom optimization.